In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from sklearn.metrics import mean_squared_error

In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(device)

In [ ]:
data = pd.read_parquet(
    "\\"
)

print(data.shape)
data.head()

In [ ]:
client_stats = (
    data
    .groupby("eup_grid_id")
    ["balance_in_euro"]
    .agg(
        mean="mean",
        std="std"
    )
)

active_clients = (
    client_stats
    .sort_values(
        "std",
        ascending=False
    )
    .head(20)
    .index
)

print(active_clients)

In [ ]:
def winkler_score(
    y_true,
    lower,
    upper,
    alpha=0.10
):

    scores = []

    for y,l,u in zip(
        y_true.flatten(),
        lower.flatten(),
        upper.flatten()
    ):

        width = u - l

        if y < l:

            score = (
                width +
                (2/alpha)*(l-y)
            )

        elif y > u:

            score = (
                width +
                (2/alpha)*(y-u)
            )

        else:

            score = width

        scores.append(score)

    return np.mean(scores)

In [ ]:
def make_windows(
    y,
    history=100,
    horizon=28
):

    X = []
    Y = []

    for i in range(
        len(y)-history-horizon+1
    ):

        X.append(
            y[i:i+history]
        )

        Y.append(
            y[
                i+history:
                i+history+horizon
            ]
        )

    return (
        np.array(X),
        np.array(Y)
    )

In [ ]:
class DeepAR(nn.Module):

    def __init__(
        self,
        hidden_size=64,
        horizon=28
    ):

        super().__init__()

        self.lstm = nn.LSTM(

            input_size=1,

            hidden_size=hidden_size,

            num_layers=1,

            batch_first=True
        )

        self.mu_head = nn.Linear(
            hidden_size,
            horizon
        )

        self.sigma_head = nn.Linear(
            hidden_size,
            horizon
        )

    def forward(
        self,
        x
    ):

        out, _ = self.lstm(x)

        h = out[:, -1]

        mu = self.mu_head(h)

        sigma = torch.exp(
            self.sigma_head(h)
        )

        return mu, sigma

In [ ]:
def gaussian_nll(
    y,
    mu,
    sigma
):

    return torch.mean(

        torch.log(sigma)

        +

        ((y-mu)**2)

        /(2*sigma**2)

    )

In [ ]:
def run_deepar_pipeline(
    series,
    dates
):

    history = 100
    horizon = 28
    epochs = 50

    y_log = np.log1p(series)

    X,Y = make_windows(
        y_log,
        history,
        horizon
    )

    if len(X) < 50:
        return None

    n = len(X)

    train_end = int(n*0.60)
    cal_end = int(n*0.80)

    X_train = X[:train_end]
    Y_train = Y[:train_end]

    X_cal = X[train_end:cal_end]
    Y_cal = Y[train_end:cal_end]

    X_test = X[cal_end:]
    Y_test = Y[cal_end:]

    mean = X_train.mean()

    std = X_train.std() + 1e-6

    X_train_s = (
        X_train - mean
    ) / std

    X_cal_s = (
        X_cal - mean
    ) / std

    X_test_s = (
        X_test - mean
    ) / std

    Y_train_s = (
        Y_train - mean
    ) / std

    Xt = torch.tensor(
        X_train_s[...,None],
        dtype=torch.float32
    ).to(device)

    Yt = torch.tensor(
        Y_train_s,
        dtype=torch.float32
    ).to(device)

    model = DeepAR(
        horizon=horizon
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=1e-3
    )

    for epoch in range(epochs):

        model.train()

        mu,sigma = model(Xt)

        loss = gaussian_nll(
            Yt,
            mu,
            sigma
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

    model.eval()

    Xc = torch.tensor(
        X_cal_s[...,None],
        dtype=torch.float32
    ).to(device)

    Xs = torch.tensor(
        X_test_s[...,None],
        dtype=torch.float32
    ).to(device)

    with torch.no_grad():

        mu_cal,sigma_cal = model(Xc)

        mu_test,sigma_test = model(Xs)

    mu_cal = mu_cal.cpu().numpy()
    sigma_cal = sigma_cal.cpu().numpy()

    mu_test = mu_test.cpu().numpy()
    sigma_test = sigma_test.cpu().numpy()

    z = 1.645

    lower_cal = (
        mu_cal
        -
        z*sigma_cal
    )

    upper_cal = (
        mu_cal
        +
        z*sigma_cal
    )

    lower_cal = lower_cal*std+mean
    upper_cal = upper_cal*std+mean

    scores = np.maximum(

        lower_cal - Y_cal,

        Y_cal - upper_cal

    )

    qhat = []

    for h in range(horizon):

        q = np.quantile(

            scores[:,h],

            0.90,

            method="higher"
        )

        qhat.append(
            max(0,q)
        )

    qhat = np.array(qhat)

    lower_test = (
        mu_test
        -
        z*sigma_test
    )

    upper_test = (
        mu_test
        +
        z*sigma_test
    )

    mu_test = (
        mu_test*std+mean
    )

    lower_test = (
        lower_test*std+mean
    )

    upper_test = (
        upper_test*std+mean
    )

    lower_conf = (
        lower_test
        -
        qhat
    )

    upper_conf = (
        upper_test
        +
        qhat
    )

    actual = np.expm1(
        Y_test
    )

    pred = np.expm1(
        mu_test
    )

    lower_conf = np.expm1(
        lower_conf
    )

    upper_conf = np.expm1(
        upper_conf
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual.flatten(),
            pred.flatten()
        )
    )

    coverage = np.mean(
        (actual>=lower_conf)
        &
        (actual<=upper_conf)
    )

    winkler = winkler_score(
        actual,
        lower_conf,
        upper_conf
    )

    return {

        "rmse": rmse,

        "coverage": coverage,

        "winkler": winkler,

        "actual": actual,

        "forecast": pred,

        "lower_conf": lower_conf,

        "upper_conf": upper_conf,

        "series": series,

        "dates": dates
    }

In [ ]:
all_results = {}
all_metrics = []

for client_id in active_clients:

    try:

        subset = (
            data[
                data["eup_grid_id"]
                == client_id
            ]
            .copy()
        )

        subset["date"] = pd.to_datetime(
            subset["date"]
        )

        subset = (
            subset
            .sort_values("date")
            .set_index("date")
            .asfreq("D")
        )

        subset["balance_in_euro"] = (
            subset["balance_in_euro"]
            .ffill()
            .fillna(0)
        )

        series = (
            subset["balance_in_euro"]
            .astype(float)
            .values
        )

        result = run_deepar_pipeline(
            series,
            subset.index
        )

        if result is None:
            continue

        all_results[client_id] = result

        all_metrics.append({

            "client_id": client_id,

            "rmse": result["rmse"],

            "coverage": result["coverage"],

            "winkler": result["winkler"],

            "std_balance": np.std(series)

        })

        print(
            "Done:",
            client_id
        )

    except Exception as e:

        print(
            client_id,
            e
        )


In [ ]:
metrics_df = pd.DataFrame(
    all_metrics
)

metrics_df["nrmse"] = (
    metrics_df["rmse"]
    /
    metrics_df["std_balance"]
)

metrics_df["nwinkler"] = (
    metrics_df["winkler"]
    /
    metrics_df["std_balance"]
)

In [ ]:
best_3 = (
    metrics_df
    .sort_values("nwinkler")
    .head(3)
)

worst_3 = (
    metrics_df
    .sort_values(
        "nwinkler",
        ascending=False
    )
    .head(3)
)

In [ ]:
def plot_deepar(
    result,
    title
):

    dates = result["dates"]

    history = result["series"][-128:-28]

    history_dates = dates[-128:-28]

    future_dates = dates[-28:]

    actual = result["actual"][-1]

    forecast = result["forecast"][-1]

    lower = result["lower_conf"][-1]

    upper = result["upper_conf"][-1]

    plt.figure(figsize=(15,6))

    plt.plot(
        history_dates,
        history,
        color="black",
        linewidth=2,
        label="History"
    )

    plt.plot(
        future_dates,
        actual,
        color="blue",
        linewidth=2,
        label="Actual"
    )

    plt.plot(
        future_dates,
        forecast,
        color="red",
        linewidth=2,
        label="Forecast"
    )

    plt.fill_between(
        future_dates,
        lower,
        upper,
        color="green",
        alpha=0.25,
        label="Conformal PI"
    )

    plt.axvline(
        history_dates[-1],
        color="red",
        linestyle="--"
    )

    plt.title(
        f"{title}\n"
        f"RMSE={result['rmse']:,.0f} | "
    f"Coverage={result['coverage']:.2%} | "
        f"Winkler={result['winkler']:,.0f}"
    )

    plt.xticks(rotation=45)

    plt.legend()

    plt.tight_layout()

    plt.show()

In [ ]:
for i, client_id in enumerate(
    best_3["client_id"],
    start=1
):
    plot_deepar(
        all_results[client_id],
        f"Best Forecast {i}"
    )

for i, client_id in enumerate(
    worst_3["client_id"],
    start=1
):
    plot_deepar(
        all_results[client_id],
        f"Worst Forecast {i}"
    )

In [ ]:
aggregate_metrics = {
    "n_clients": len(metrics_df),

    "mean_nrmse": metrics_df["nrmse"].mean(),
    "median_nrmse": metrics_df["nrmse"].median(),

    "mean_coverage": metrics_df["coverage"].mean(),
    "median_coverage": metrics_df["coverage"].median(),

    "mean_nwinkler": metrics_df["nwinkler"].mean(),
    "median_nwinkler": metrics_df["nwinkler"].median()
}

print("\nAggregate DeepAR Results")
print("------------------------")

for key, value in aggregate_metrics.items():
    print(f"{key}: {value}")